# 05. Prueba de los controles de calidad de CAMMESA

Prueba de la detección de meses faltantes y de meses revisados entre ediciones (tarea 6 del backlog, HU-02). Las dos marcas se calculan en `src/carga_cammesa.py` y se muestran en el informe de calidad de la pantalla de carga.

In [1]:
import sys
sys.path.append("../src")

In [2]:
import pandas as pd
from pathlib import Path
from carga_cammesa import union_filtro
from carga_cammesa import meses_faltantes
from carga_cammesa import apilar_ediciones
from carga_cammesa import detectar_revisados
from carga_cammesa import deduplicar
from carga_cammesa import csv_tabla

## 1. Períodos faltantes

`meses_faltantes` arma todos los meses entre el primero y el último de la serie y devuelve los que no están. Para verificar que detecta un hueco, se elimina un mes de una copia de la tabla.

In [3]:
rutas = sorted(Path("../data/raw/cammesa").glob("CAMMESA_MEM_ReqMax_*.xlsx"))
tabla = union_filtro(rutas)

In [4]:
tabla_hueco = tabla.drop(index=10)

In [5]:
ruta_temp = Path("../data/processed/OPENMETEO_TRELEW_TempMensual.csv")
temperatura = pd.read_csv(ruta_temp, parse_dates=["MES"])
meses_faltantes(temperatura)

DatetimeIndex([], dtype='datetime64[us]', freq='MS')

In [6]:
print("Tabla completa: ",meses_faltantes(tabla))
print("Tabla con faltantes: ",meses_faltantes(tabla_hueco))

Tabla completa:  DatetimeIndex([], dtype='datetime64[us]', freq='MS')
Tabla con faltantes:  DatetimeIndex(['2019-11-01'], dtype='datetime64[us]', freq='MS')


## 2. Ediciones y meses revisados

CAMMESA no indica qué valores son provisorios, pero sí corrige meses ya publicados en ediciones posteriores. `apilar_ediciones` junta los registros de todas las ediciones sin descartar ninguno, y `detectar_revisados` marca los meses cuyo máximo real cambió más de 0,0005 MW entre una edición y otra. Después, `deduplicar` conserva el valor de la edición más nueva.

In [7]:
ediciones = apilar_ediciones(rutas)
print(len(ediciones))

189


In [8]:
revisados = detectar_revisados(ediciones)
print(sum(revisados))

36


In [9]:
serie = deduplicar(ediciones)
len(serie)

93

In [10]:
serie["EDICION"].value_counts()

EDICION
2026-09    45
2024-12    24
2022-12    12
2023-12    12
Name: count, dtype: int64

## 3. Serie final

`union_filtro` hace los tres pasos en uno y agrega la columna `REVISADO` a la serie.

In [11]:
tabla.info()
tabla["REVISADO"].sum()

<class 'pandas.DataFrame'>
RangeIndex: 93 entries, 0 to 92
Data columns (total 7 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   MES                        93 non-null     datetime64[us]
 1   MAX_REAL_MW                93 non-null     float64       
 2   POTENCIA_DECLARADA_MW      93 non-null     float64       
 3   EDICION                    93 non-null     str           
 4   DESCARGA                   93 non-null     str           
 5   REVISADO                   93 non-null     bool          
 6   POSIBLE_COPIA_DECLARACION  93 non-null     bool          
dtypes: bool(2), datetime64[us](1), float64(2), str(2)
memory usage: 5.5 KB


np.int64(36)

In [12]:
ruta_salida = Path("../data/processed/CAMMESA_CTRELEUW_ReqMax.csv")
csv_tabla(tabla, ruta_salida)

## Conclusión

- La serie no tiene meses faltantes ni en la demanda ni en la temperatura. El control detecta un hueco cuando se lo provoca.
- Las cuatro ediciones suman 189 registros del agente, que se reducen a 93 meses.
- 36 meses tienen un valor distinto entre ediciones. Son la evidencia de que CAMMESA corrige datos ya publicados, y por lo mismo la marca reemplaza la regla de "los dos meses más recientes son provisorios" planteada en el Entregable 3. Su límite es que solo detecta un mes después de que fue corregido.